# <center>
<div style="
    background: #f0f333ff;
    border-left: 5px solid #ecd242ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h1 style="
        text-align: center;
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 600;
    ">
    Nettoyage
    </h1>
</div>

##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Question
    </h2>
</div>

- Intégrer et tester le module fourni "validation_structure"
- Décider ce qu'est un doublon
- Appeler le service sur une vingtaine de numéros de pays diférents

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Imports
    </h3>
</div>

In [140]:
import os
import sys
import time
import requests

import pandas as pd
import matplotlib.pyplot as plt
# On importe le module

sys.path.append('../api')
from validation_structure import normaliser, valider

# Reading Data
df_csv = pd.read_csv('../data/numeros_tva.csv', index_col='id', sep=',', keep_default_na = False)
df_xl  = pd.read_excel('../data/numeros_tva.xlsx', index_col='id', keep_default_na = False)

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Utilisation des fonctions "Normaliser" et "Valider" du module "validation_structure.py"
    </h3>
</div>

In [141]:
# Normalisation des numéros
df_csv['numero_norm'] = df_csv['numero_tva'].apply(normaliser)

# Validation des numéros
resultats = df_csv['numero_norm'].apply(valider)

df_csv[['Verdict', 'Motif']] = pd.DataFrame(
    resultats.tolist(),
    index=df_csv.index,
    columns=['Verdict', 'Motif']
)

df_csv.sample(10)

,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
55,Verso Foods Lda,LU,LU84259936,2024-06-12,reprise_erp,LU84259936,True,ok
5391,Nord Logistics AB,FR,FR89298264388,2025-01-22,portail_client,FR89298264388,True,ok
4724,Vela Distribution SpA,DK,78883464,2024-07-14,portail_client,78883464,False,pays_non_couvert
8866,Global Foods SA,LU,LU31271255,2024-10-11,crm,LU31271255,True,ok
5855,Global Technologies SA,PT,PT8153023G0,2024-08-25,crm,PT8153023G0,False,cle_invalide
1328,Cobalt Logistics NV,LU,LU64325300,2024-09-14,reprise_erp,LU64325300,False,cle_invalide
1693,Kappa Trading Sp. z o.o.,FR,FR16801844746,2024-09-08,portail_client,FR16801844746,True,ok
4561,Lumen Technologies Lda,FI,FI6427400,2025-08-13,portail_client,FI6427400,False,cle_invalide
3883,Baltic Materials GmbH,PT,N/A,2024-08-15,crm,NA,False,trop_court


Motifs possibles : 

| Motif | Explication |
|:--:|:--:|
| `ok` | Le numéro de TVA est valide |
| `vide` | Le numéro de TVA est vide |
| `trop_court` | Le numéro est trop court (moins de 4 caractères) donc invalide |
| `pays_non_couvert` | L'indicatif du pays du numéro de TAV n'est pas pris en compte |
| `cle_invalide` | La clé du numéro de TAV est invalide |

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Comptage du nombre de numéros valides et des motifs
    </h3>
</div>

In [142]:
print('-' * 8 + ' Après avoir normalisé les numéros ' + '-' * 8 + '\n')
ds = df_csv['Verdict'].value_counts()
print(f'Nombre de numéros valides   : {ds.values[0]}')
print(f'Nombre de numéros invalides : {ds.values[1]}\n')

ds = df_csv['Motif'].value_counts()
print("Motifs des invalides :")
print(f'Nombre de numéros avec une clé invalide : {ds.values[1]}')
print(f"Nombre de numéros d'un pays non couvert : {ds.values[2]}")
print(f'Nombre de numéros vides                 : {ds.values[3]}')
print(f'Nombre de numéros trop courts           : {ds.values[4]}')
print(f'Total \t\t\t\t\t: {sum([ds.values[i] for i in range(1, 5)])}')

-------- Après avoir normalisé les numéros --------

Nombre de numéros valides   : 6083
Nombre de numéros invalides : 3917

Motifs des invalides :
Nombre de numéros avec une clé invalide : 2605
Nombre de numéros d'un pays non couvert : 1090
Nombre de numéros vides                 : 174
Nombre de numéros trop courts           : 48
Total 					: 3917


##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Recherche des doublons
    </h2>
</div>

Décidez ensuite de ce qu'est un doublon dans ce référentiel, et comptez-les. Votre définition doit pouvoir s'écrire en une phrase.

Nous allons chercher les doublons sur :
- Les numéros de TVA seuls
- Les couples : (numéros de TVA, raison sociale)
- Les lignes entières

####
<div style="
    background: #eebbefff;
    border-left: 5px solid #d619efff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 400;
    ">
    Doublon sur les lignes entières
    </h3>
</div>

In [143]:
print(f'Il y a {df_csv[df_csv.duplicated()].shape[0]} doublons pour les lignes entières')

Il y a 0 doublons pour les lignes entières


####
<div style="
    background: #eebbefff;
    border-left: 5px solid #d619efff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 400;
    ">
    Doublon sur les couples : (numéros de TVA, raison sociale)
    </h3>
</div>

In [144]:
# Doublon sur les numéros de TVA
df_duplicate = df_csv[df_csv[['numero_tva', 'raison_sociale']].duplicated()].sort_values(by='numero_tva')
print(f'Il y a {df_duplicate.shape[0]} doublons pour les couples (numero_tva, raison_sociale)')
display(df_duplicate.value_counts(['numero_tva', 'raison_sociale']))
df_duplicate

Il y a 147 doublons pour les couples (numero_tva, raison_sociale)


numero_tva      raison_sociale          
0282330178      Papyrus Industries SA       3
PT823046656     Kappa Foods SAS             2
                Vela Consulting SA          1
                Orion Distribution Oy       1
                Vela Materials Lda          1
                                           ..
nl442736873b35  Ardent Distribution SA      1
nl785427788b39  Vela Materials SpA          1
pl1524029668    Prima Foods NV              1
pl1887742564    Lusitania Consulting Lda    1
se740563530601  Hanse Transports A/S        1
Name: count, Length: 144, dtype: int64

,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
4158,Vela Consulting SA,IT,,2024-03-31,portail_client,,False,vide
3205,Orion Distribution Oy,BE,,2024-10-22,crm,,False,vide
9184,Vela Materials Lda,DK,,2025-07-10,crm,,False,vide
4473,Corvus Trading Lda,IT,,2025-08-19,portail_client,,False,vide
5386,Kappa Distribution SARL,FI,,2025-09-09,portail_client,,False,vide
...,...,...,...,...,...,...,...,...
9852,Ardent Distribution SA,NL,nl442736873b35,2025-09-03,saisie_manuelle,NL442736873B35,True,ok
760,Vela Materials SpA,NL,nl785427788b39,2024-02-23,crm,NL785427788B39,True,ok
7822,Prima Foods NV,PL,pl1524029668,2025-04-10,saisie_manuelle,PL1524029668,True,ok


####
<div style="
    background: #eebbefff;
    border-left: 5px solid #d619efff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 400;
    ">
    Doublon sur les numéros de TVA seuls
    </h3>
</div>

In [145]:
df_duplicate = df_csv[df_csv['numero_tva'].duplicated()]
print(f'Il y a {df_duplicate.shape[0]} doublons pour les numéros de TVA seuls')
display(df_duplicate.sort_values(by='numero_tva'))
df_duplicate['numero_tva'].value_counts()

Il y a 390 doublons pour les numéros de TVA seuls


,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
1839,Lumen Services NV,DK,,2025-01-12,saisie_manuelle,,False,vide
2175,Kappa Services NV,PT,,2024-04-27,saisie_manuelle,,False,vide
7056,Verso Consulting SA,FR,,2025-03-06,crm,,False,vide
4867,Lumen Technologies GmbH,DK,,2025-01-27,reprise_erp,,False,vide
8040,Verso Industries Lda,FI,,2024-11-13,crm,,False,vide
...,...,...,...,...,...,...,...,...
9153,Lumen Services Sp. z o.o.,PL,null,2024-12-28,import_fournisseur,NULL,False,pays_non_couvert
9367,Corvus Distribution BV,LU,null,2025-07-19,portail_client,NULL,False,pays_non_couvert
7822,Prima Foods NV,PL,pl1524029668,2025-04-10,saisie_manuelle,PL1524029668,True,ok


numero_tva
                  59
                  58
-                 54
N/A               47
null              37
                  ..
PL3605374689       1
ITJ5457240285      1
nl442736873b35     1
BE 1769 956921     1
FR60801776699      1
Name: count, Length: 137, dtype: int64

In [146]:
df_csv[(df_csv['numero_tva'] == '0282330178') & (df_csv['raison_sociale'] =='Papyrus Industries SA')]
df_csv[df_csv['numero_tva'] == 'ITJ5457240285']

,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
2265,Kappa Transports Sp. z o.o.,IT,ITJ5457240285,2024-09-18,reprise_erp,ITJ5457240285,False,cle_invalide
9834,Kappa Transports Sp. z o.o.,IT,ITJ5457240285,2025-07-31,import_fournisseur,ITJ5457240285,False,cle_invalide


**Conclusion**

Nous choisirons comme critère de "doublon" les numeros de TAV seuls car notre but est de déterminer quels numéros sont valides ou non valides et non pas de savoir si les numéros sont rattachés à la bonne société ou si la source est bonne.

In [147]:
print(f'Taille du dataset AVANT suppression des doublons : {df_csv.shape[0]}')
# df_no_duplicate = df_csv.drop_duplicates(subset=['numero_tva', 'raison_sociale'])
df_no_duplicate = df_csv.drop_duplicates(subset=['numero_tva'])
print(f'Taille du dataset APRES suppression des doublons : {df_no_duplicate.shape[0]}')

Taille du dataset AVANT suppression des doublons : 10000
Taille du dataset APRES suppression des doublons : 9610


In [148]:
print('-' * 8 + ' Après avoir normalisé les numéros sans doublon ' + '-' * 8 + '\n')
ds = df_no_duplicate['Verdict'].value_counts()
print(f'Nombre de numéros valides   : {ds.values[0]}')
print(f'Nombre de numéros invalides : {ds.values[1]}\n')

ds = df_no_duplicate['Motif'].value_counts()
print("Motifs des invalides :")
print(f'Nombre de numéros avec une clé invalide : {ds.values[1]}')
print(f"Nombre de numéros d'un pays non couvert : {ds.values[2]}")
print(f'Nombre de numéros vides                 : {ds.values[3]}')
print(f'Nombre de numéros trop courts           : {ds.values[4]}')
print(f'Total \t\t\t\t\t: {sum([ds.values[i] for i in range(1, 5)])}')

-------- Après avoir normalisé les numéros sans doublon --------

Nombre de numéros valides   : 5998
Nombre de numéros invalides : 3612

Motifs des invalides :
Nombre de numéros avec une clé invalide : 2572
Nombre de numéros d'un pays non couvert : 1036
Nombre de numéros vides                 : 3
Nombre de numéros trop courts           : 1
Total 					: 3612


##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Appel du service API (benchmark)
    </h2>
</div>

Nous allons lancer le service sur quelques numéros pour mesurer le temps de réponse.

In [ ]:
size_sample = 20
lst_sample = df_no_duplicate[['numero_norm', 'Verdict']].sample(size_sample).values.tolist()

lst_countries, lst_numeros, lst_script_verdict, lst_api_verdict, lst_api_time = [], [], [], [], []
# lst_numeros_to_check = []

for item in lst_sample:
    code_pays, numero = item[0][:2], item[0][2:]
    start = time.time()
    try:
        r = requests.get(f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{code_pays}/vat/{numero}", timeout=10)
        elapsed_time = time.time() - start
        if r.status_code == 200:
            lst_countries.append(code_pays)
            lst_numeros.append(numero)
            lst_script_verdict.append(item[1])
            lst_api_verdict.append("VALID" if r.json()['isValid'] else "INVALID")
            lst_api_time.append(elapsed_time)

            # if r.json()['isValid'] and item[1] != 'VALID':
            #     # Si l'API renvoie VALID mais que notre script renvoie INVALID
            #     lst_numeros_to_check.append([code_pays, numero])

    except requests.exceptions.Timeout:
        print(f'Délai dépassé : {item[0]}')

df_test = pd.DataFrame({
})

df_results = pd.DataFrame({
    'Pays' : lst_countries,
    'Numero' : lst_numeros,
    'script_verdict' : lst_script_verdict,
    'api_verdict' : lst_api_verdict,
    'api_time' : lst_api_time
})

# df_numero_to_check = pd.DataFrame({
#     'Pays'   : [item[0] for item in lst_numeros_to_check],
#     'numero' : [item[1] for item in lst_numeros_to_check]
# })

print(f'Nombre de requêtes : {size_sample}')
print(f'Moyenne du temps de requête : {sum(lst_api_time) / size_sample}')
display(df_results)
# df_numero_to_check